# Limpieza de dataset - Fases RBC (QPI)

Issue: [#7](https://github.com/Jacogz/DL-QPI/issues/7)

Objetivo: Aplicar limpieza al dataset de fases RBC.

## Criterio acordado
- Eliminación de imágenes con manchas oscuras.


## 1. Configuración y carga de datos

In [ ]:
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2

random.seed(123)
np.random.seed(123)

cwd = Path.cwd()
REPO_ROOT = cwd
while REPO_ROOT != REPO_ROOT.parent:
    if REPO_ROOT.name == 'DL-QPI' or (REPO_ROOT / 'pyproject.toml').exists() or (REPO_ROOT / '.git').exists():
        break
    REPO_ROOT = REPO_ROOT.parent

CANDIDATE_DIRS = [
    REPO_ROOT / 'data' / 'raw' / 'rbc' / 'Phase',
    REPO_ROOT / 'data' / 'raw' / 'erythrocytes' / 'Phase',
    Path('/content/data/rbc_holograms/Phase'),
    Path('/content/data/rbc_holograms'),
]

DATA_DIR = None
for d in CANDIDATE_DIRS:
    if d.exists():
        if (d / 'Phase').exists():
            cand = d / 'Phase'
            DATA_DIR = cand / 'Training' if (cand / 'Training').exists() else cand
        elif (d / 'Training').exists():
            DATA_DIR = d / 'Training'
        else:
            DATA_DIR = d
        break

OUT_DIR = REPO_ROOT / 'data' / 'processed' / 'rbc_fases'
OUT_DIR.mkdir(parents=True, exist_ok=True)

if DATA_DIR is None:
    print('No se encontró el dataset de fases RBC. Coloca los archivos en:')
    for d in CANDIDATE_DIRS:
        print(f' - {d}')
    raise FileNotFoundError('Dataset de fases RBC no encontrado')

print(f'REPO_ROOT : {REPO_ROOT}')
print(f'DATA_DIR  : {DATA_DIR}')
print(f'Salida    : {OUT_DIR.resolve()}')

In [ ]:
IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}
image_files = sorted([f for f in Path(DATA_DIR).rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS])
print(f'Imágenes encontradas: {len(image_files)}')

## 2. Detección de manchas oscuras

In [ ]:
def detect_dark_spots(img_arr, dark_thresh=30, min_area_ratio=0.01):
    if img_arr.ndim == 3:
        gray = cv2.cvtColor(img_arr, cv2.COLOR_RGB2GRAY)
    else:
        gray = img_arr
    h, w = gray.shape
    total = h * w
    _, mask = cv2.threshold(gray, dark_thresh, 255, cv2.THRESH_BINARY_INV)
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    dark_area = 0
    large_spots = 0
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        dark_area += area
        if area / total > 0.005:
            large_spots += 1
    area_ratio = dark_area / total if total > 0 else 0
    mean_val = float(gray.mean())
    has_dark_spots = (area_ratio > min_area_ratio) or (mean_val < 40) or (large_spots >= 2)
    return has_dark_spots, {'area_ratio': area_ratio, 'mean_val': mean_val, 'large_spots': large_spots}

## 3. Aplicar filtro y conteo

In [ ]:
removal_log = defaultdict(int)
kept_images = []

for img_path in image_files:
    try:
        img = Image.open(img_path).convert('L')
        arr = np.array(img)
        has_spots, stats = detect_dark_spots(arr)
        if has_spots:
            removal_log['manchas_oscuras'] += 1
        else:
            kept_images.append(img_path)
    except Exception as e:
        removal_log['errores_lectura'] += 1

print('Conteo de imágenes eliminadas por criterio:')
for k, v in removal_log.items():
    print(f'  {k}: {v}')
print(f'\nImágenes mantenidas: {len(kept_images)}')

## 4. Ejemplos antes/después

In [ ]:
candidates = []
for img_path in random.sample(image_files, min(20, len(image_files))) if image_files else []:
    try:
        img = Image.open(img_path).convert('L')
        arr = np.array(img)
        has_spots, stats = detect_dark_spots(arr)
        if has_spots:
            candidates.append((img_path, arr, stats))
            if len(candidates) >= 3:
                break
    except Exception:
        pass

if candidates:
    n_show = len(candidates)
    fig, axes = plt.subplots(n_show, 2, figsize=(10, 4*n_show))
    if n_show == 1:
        axes = np.array([axes])
    for idx, (img_path, arr, stats) in enumerate(candidates):
        axes[idx, 0].imshow(arr, cmap='gray')
        axes[idx, 0].set_title(f'Antes - {img_path.name}\nmean={stats["mean_val"]:.1f}, dark_area={stats["area_ratio"]:.3f}')
        axes[idx, 1].text(0.5, 0.5, 'ELIMINADA\n(manchas oscuras)', ha='center', va='center', fontsize=14)
        axes[idx, 1].axis('off')
    for ax in axes[:, 0]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No se encontraron ejemplos con manchas oscuras en la muestra.')

if kept_images:
    n_show_k = min(3, len(kept_images))
    fig, axes = plt.subplots(1, n_show_k, figsize=(12, 4))
    if n_show_k == 1:
        axes = np.array([axes])
    for idx, img_path in enumerate(kept_images[:n_show_k]):
        img = Image.open(img_path).convert('L')
        axes[idx].imshow(img, cmap='gray')
        axes[idx].set_title(f'Mantenida - {img_path.name}')
        axes[idx].axis('off')
    plt.tight_layout()
    plt.show()

## 5. Guardar dataset limpio

In [ ]:
count_saved = 0
for img_path in kept_images:
    try:
        img = Image.open(img_path).convert('L')
        img.save(OUT_DIR / img_path.name)
        count_saved += 1
    except Exception as e:
        print(f'Error guardando {img_path.name}: {e}')

print(f'Imágenes guardadas: {count_saved}')
print(f'Directorio: {OUT_DIR.resolve()}')